In [ ]:
!pip install medmnist


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [medmnist]


In [ ]:
import os


DATA_ROOT = "/content/medmnist_data"
os.makedirs(DATA_ROOT, exist_ok=True)


print("Dataset directory:", DATA_ROOT)
print("Directory exists:", os.path.exists(DATA_ROOT))


Dataset directory: /content/medmnist_data
Directory exists: True


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/MedicalImageAI/LossComparison"

os.makedirs(BASE_DIR, exist_ok=True)

print("Save directory:", BASE_DIR)
print("Directory exists:", os.path.exists(BASE_DIR))

Save directory: /content/drive/MyDrive/MedicalImageAI/LossComparison
Directory exists: True


In [ ]:

# ============================================================
# RESNET18 - LOSS FUNCTION COMPARISON
# 1. BCE
# 2. Square-root Weighted BCE
# 3. Weighted BCE
# 4. Asymmetric Loss (ASL)
# ============================================================

import os
import json
import random
import gc
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from contextlib import nullcontext
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score
)
from medmnist import ChestMNIST

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

SEED = 42

BATCH_SIZE = 32
NUM_WORKERS = 2

MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
MIN_DELTA = 0.0001

LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.0001

OUTPUT_DIR = "/content/resnet18_loss_comparison"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)
print("Mixed precision:", USE_AMP)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)


# ------------------------------------------------------------
# 2. LOAD DATASETS
# ------------------------------------------------------------

if "raw_train" not in globals() or not hasattr(raw_train, "imgs"):
    raw_train = ChestMNIST(
      split="train",
      download=True,
      size=128,
      root=DATA_ROOT
    )

if "raw_val" not in globals() or not hasattr(raw_val, "imgs"):
    raw_val = ChestMNIST(
      split="val",
      download=True,
      size=128,
      root=DATA_ROOT
    )
print("\nDataset sizes:")
print("Train:", len(raw_train.imgs))
print("Validation:", len(raw_val.imgs))


# Six selected labels
SELECTED_INDICES = [0, 1, 2, 3, 4, 7]

CLASS_NAMES = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Pneumothorax"
]

train_labels = np.asarray(raw_train.labels)[:, SELECTED_INDICES].astype(
    np.float32
)

val_labels = np.asarray(raw_val.labels)[:, SELECTED_INDICES].astype(
    np.float32
)

assert train_labels.shape[1] == 6
assert val_labels.shape[1] == 6

print("\nSelected classes:", CLASS_NAMES)


# ------------------------------------------------------------
# 3. CALCULATE POSITIVE CLASS WEIGHTS
# ------------------------------------------------------------

positive_counts = train_labels.sum(axis=0)
negative_counts = len(train_labels) - positive_counts

pos_weights_np = negative_counts / np.maximum(positive_counts, 1)

pos_weights_np = np.clip(
    pos_weights_np,
    1.0,
    50.0
)

print("\nPositive weights:")

for name, weight in zip(CLASS_NAMES, pos_weights_np):
    print(f"{name}: {weight:.4f}")

POS_WEIGHTS = torch.tensor(
    pos_weights_np,
    dtype=torch.float32,
    device=DEVICE
)

SQRT_POS_WEIGHTS = torch.sqrt(POS_WEIGHTS)


# ------------------------------------------------------------
# 4. DATASET AND TRANSFORMS
# ------------------------------------------------------------

# Keep the original regularization augmentation.
# No horizontal flip is used.

train_transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=7,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        interpolation=InterpolationMode.BILINEAR,
        fill=0
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])


class SelectedLabelsDataset(Dataset):

    def __init__(self, base_dataset, transform=None):

        self.images = base_dataset.imgs

        self.labels = np.asarray(
            base_dataset.labels
        )[:, SELECTED_INDICES].astype(np.float32)

        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):

        image_array = np.asarray(self.images[index])

        if image_array.ndim == 3 and image_array.shape[-1] == 1:
            image_array = image_array.squeeze(-1)

        if image_array.dtype != np.uint8:
            image_array = np.clip(
                image_array, 0, 255
            ).astype(np.uint8)

        image = Image.fromarray(image_array).convert("L")

        if self.transform is not None:
            image = self.transform(image)

        label = torch.tensor(
            self.labels[index],
            dtype=torch.float32
        )

        return image, label


train_dataset = SelectedLabelsDataset(
    raw_train,
    transform=train_transform
)

val_dataset = SelectedLabelsDataset(
    raw_val,
    transform=val_transform
)


def seed_worker(worker_id):

    worker_seed = torch.initial_seed() % (2 ** 32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def create_loaders():

    # Same shuffle and augmentation random seed for each experiment.
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(NUM_WORKERS > 0)
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        worker_init_fn=seed_worker,
        persistent_workers=(NUM_WORKERS > 0)
    )

    return train_loader, val_loader


# ------------------------------------------------------------
# 5. CREATE COMMON IMAGENET INITIALIZATION
# ------------------------------------------------------------

print("\nLoading ImageNet pretrained ResNet18...")

set_seed(SEED)

initial_model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Adapt RGB input to grayscale.
original_conv = initial_model.conv1

new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

with torch.no_grad():
    new_conv.weight.copy_(
        original_conv.weight.mean(dim=1, keepdim=True)
    )

initial_model.conv1 = new_conv

# Adapt output layer to six classes.
initial_model.fc = nn.Linear(
    initial_model.fc.in_features,
    6
)

# Store exactly the same initialization for all four experiments.
INITIAL_STATE = {
    key: value.detach().cpu().clone()
    for key, value in initial_model.state_dict().items()
}

del initial_model
gc.collect()

print("Common initialization prepared!")


def create_model():

    model = models.resnet18(weights=None)

    model.conv1 = nn.Conv2d(
        1, 64,
        kernel_size=7,
        stride=2,
        padding=3,
        bias=False
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        6
    )

    model.load_state_dict(INITIAL_STATE)

    return model.to(DEVICE)


# ------------------------------------------------------------
# 6. ASYMMETRIC LOSS
# ------------------------------------------------------------

class AsymmetricLoss(nn.Module):

    def __init__(
        self,
        gamma_neg=4,
        gamma_pos=1,
        clip=0.05,
        eps=1e-8
    ):

        super().__init__()

        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):

        probabilities = torch.sigmoid(logits)

        prob_positive = probabilities
        prob_negative = 1.0 - probabilities

        # Clip negative probabilities.
        if self.clip is not None and self.clip > 0:
            prob_negative = (
                prob_negative + self.clip
            ).clamp(max=1.0)

        # Log likelihood.
        loss = (
            targets * torch.log(
                prob_positive.clamp(min=self.eps)
            )
            +
            (1.0 - targets) * torch.log(
                prob_negative.clamp(min=self.eps)
            )
        )

        # Asymmetric focusing.
        if self.gamma_neg > 0 or self.gamma_pos > 0:

            pt = (
                prob_positive * targets
                +
                prob_negative * (1.0 - targets)
            )

            gamma = (
                self.gamma_pos * targets
                +
                self.gamma_neg * (1.0 - targets)
            )

            weight = (1.0 - pt).pow(gamma)

            loss = loss * weight

        return -loss.mean()


# ------------------------------------------------------------
# 7. LOSS FUNCTION FACTORY
# ------------------------------------------------------------

def create_loss(loss_name):

    if loss_name == "bce":

        return nn.BCEWithLogitsLoss()

    elif loss_name == "sqrt_weighted_bce":

        return nn.BCEWithLogitsLoss(
            pos_weight=SQRT_POS_WEIGHTS
        )

    elif loss_name == "weighted_bce":

        return nn.BCEWithLogitsLoss(
            pos_weight=POS_WEIGHTS
        )

    elif loss_name == "asymmetric_loss":

        return AsymmetricLoss(
            gamma_neg=4,
            gamma_pos=1,
            clip=0.05
        )

    else:
        raise ValueError("Unknown loss function")


# ------------------------------------------------------------
# 8. METRICS
# ------------------------------------------------------------

def calculate_metrics(targets, probabilities):

    predictions = (probabilities >= 0.5).astype(int)

    class_aucs = {}
    class_aps = {}

    auc_values = []
    ap_values = []

    for index, class_name in enumerate(CLASS_NAMES):

        y_true = targets[:, index]
        y_prob = probabilities[:, index]

        if len(np.unique(y_true)) == 2:

            auc = roc_auc_score(y_true, y_prob)
            auc_values.append(auc)

        else:
            auc = float("nan")

        ap = average_precision_score(y_true, y_prob)
        ap_values.append(ap)

        class_aucs[class_name] = float(auc)
        class_aps[class_name] = float(ap)

    macro_auc = float(np.nanmean(auc_values))
    macro_map = float(np.mean(ap_values))

    micro_f1 = f1_score(
        targets,
        predictions,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "macro_auroc": macro_auc,
        "macro_map": macro_map,
        "micro_f1": float(micro_f1),
        "macro_f1": float(macro_f1),
        "class_auroc": class_aucs,
        "class_ap": class_aps
    }


# ------------------------------------------------------------
# 9. VALIDATION
# ------------------------------------------------------------

def evaluate_model(model, loader, criterion):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_logits = []
    all_targets = []

    with torch.no_grad():

        for images, targets in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            targets = targets.to(
                DEVICE,
                non_blocking=True
            )

            context = (
                torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                )
                if USE_AMP
                else nullcontext()
            )

            with context:

                logits = model(images)
                loss = criterion(logits, targets)

            batch_size = targets.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_logits.append(
                logits.float().cpu().numpy()
            )

            all_targets.append(
                targets.cpu().numpy()
            )

    average_loss = total_loss / total_samples

    logits = np.concatenate(all_logits, axis=0)
    targets = np.concatenate(all_targets, axis=0)

    probabilities = 1.0 / (
        1.0 + np.exp(-np.clip(logits, -50, 50))
    )

    metrics = calculate_metrics(
        targets.astype(int),
        probabilities
    )

    return average_loss, metrics, probabilities, targets


# ------------------------------------------------------------
# 10. SAVE HELPERS
# ------------------------------------------------------------

def save_json(data, path):

    with open(path, "w") as file:

        json.dump(
            data,
            file,
            indent=4
        )


def save_checkpoint(model, path, epoch, metric_name, metric_value):

    checkpoint = {
        "model_state_dict": {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        },
        "architecture": "ResNet18",
        "num_classes": 6,
        "class_names": CLASS_NAMES,
        "selected_indices": SELECTED_INDICES,
        "input_size": 128,
        "normalization": {
            "mean": [0.5],
            "std": [0.5]
        },
        "epoch": epoch,
        "selection_metric": metric_name,
        "selection_value": float(metric_value)
    }

    torch.save(checkpoint, path)


# ------------------------------------------------------------
# 11. TRAIN ONE EXPERIMENT
# ------------------------------------------------------------

def run_experiment(loss_name):

    print("\n" + "=" * 65)
    print("STARTING:", loss_name.upper())
    print("=" * 65)

    set_seed(SEED)

    experiment_dir = os.path.join(
        OUTPUT_DIR,
        loss_name
    )

    os.makedirs(experiment_dir, exist_ok=True)

    train_loader, val_loader = create_loaders()

    model = create_model()

    criterion = create_loss(loss_name)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # Scheduler monitors the same metric for every loss.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1,
        min_lr=1e-6
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    best_auc = -float("inf")
    best_map = -float("inf")

    best_auc_epoch = 0
    best_map_epoch = 0

    best_auc_metrics = None
    best_map_metrics = None

    best_auc_loss = None

    best_auc_probabilities = None
    best_map_probabilities = None

    history = []

    patience_counter = 0

    # --------------------------------------------------------
    # EPOCH LOOP
    # --------------------------------------------------------

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        running_loss = 0.0
        total_samples = 0

        for images, targets in train_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            targets = targets.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)

            context = (
                torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                )
                if USE_AMP
                else nullcontext()
            )

            with context:

                logits = model(images)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            batch_size = targets.size(0)

            running_loss += loss.item() * batch_size
            total_samples += batch_size

        train_loss = running_loss / total_samples

        # Validation
        val_loss, metrics, probabilities, targets = evaluate_model(
            model,
            val_loader,
            criterion
        )

        val_auc = metrics["macro_auroc"]
        val_map = metrics["macro_map"]

        current_lr = optimizer.param_groups[0]["lr"]

        # Scheduler uses macro AUROC.
        scheduler.step(val_auc)

        # Track the best AUROC checkpoint.
        improved_auc = val_auc > best_auc + MIN_DELTA

        if improved_auc:

            best_auc = val_auc
            best_auc_epoch = epoch

            best_auc_metrics = copy.deepcopy(metrics)
            best_auc_loss = val_loss

            best_auc_probabilities = probabilities.copy()

            save_checkpoint(
                model,
                os.path.join(
                    experiment_dir,
                    "best_model.pth"
                ),
                epoch,
                "macro_auroc",
                best_auc
            )

            np.save(
                os.path.join(
                    experiment_dir,
                    "val_probabilities.npy"
                ),
                best_auc_probabilities
            )

            np.save(
                os.path.join(
                    experiment_dir,
                    "val_targets.npy"
                ),
                targets
            )

        # Track the best mAP checkpoint separately.
        improved_map = val_map > best_map + MIN_DELTA

        if improved_map:

            best_map = val_map
            best_map_epoch = epoch

            best_map_metrics = copy.deepcopy(metrics)
            best_map_probabilities = probabilities.copy()

            save_checkpoint(
                model,
                os.path.join(
                    experiment_dir,
                    "best_model_by_map.pth"
                ),
                epoch,
                "macro_map",
                best_map
            )

            np.save(
                os.path.join(
                    experiment_dir,
                    "val_probabilities_by_map.npy"
                ),
                best_map_probabilities
            )

        # Early stopping: stop only when BOTH metrics
        # have stopped improving.
        if improved_auc or improved_map:
            patience_counter = 0
        else:
            patience_counter += 1

        epoch_result = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "macro_auroc": float(val_auc),
            "macro_map": float(val_map),
            "micro_f1": metrics["micro_f1"],
            "macro_f1": metrics["macro_f1"],
            "learning_rate": float(current_lr)
        }

        history.append(epoch_result)

        # Save progress after every epoch.
        pd.DataFrame(history).to_csv(
            os.path.join(
                experiment_dir,
                "training_history.csv"
            ),
            index=False
        )

        save_json(
            history,
            os.path.join(
                experiment_dir,
                "training_history.json"
            )
        )

        print(
            f"Epoch [{epoch}/{MAX_EPOCHS}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Macro AUROC: {val_auc:.4f} | "
            f"Macro mAP: {val_map:.4f} | "
            f"Micro F1: {metrics['micro_f1']:.4f} | "
            f"Macro F1: {metrics['macro_f1']:.4f} | "
            f"LR: {current_lr:.6f}"
        )

        if patience_counter >= EARLY_STOPPING_PATIENCE:

            print(
                "\nEarly stopping triggered."
            )

            break

    # --------------------------------------------------------
    # EXPERIMENT SUMMARY
    # --------------------------------------------------------

    summary = {
        "loss_function": loss_name,
        "seed": SEED,
        "best_auroc_epoch": best_auc_epoch,
        "best_macro_auroc": best_auc,
        "val_loss_at_best_auroc": best_auc_loss,
        "map_at_best_auroc": best_auc_metrics["macro_map"],
        "micro_f1_at_best_auroc": best_auc_metrics["micro_f1"],
        "macro_f1_at_best_auroc": best_auc_metrics["macro_f1"],
        "best_map_epoch": best_map_epoch,
        "best_macro_map": best_map,
        "auroc_at_best_map": best_map_metrics["macro_auroc"],
        "class_auroc_at_best_auroc": best_auc_metrics["class_auroc"],
        "class_ap_at_best_auroc": best_auc_metrics["class_ap"],
        "class_auroc_at_best_map": best_map_metrics["class_auroc"],
        "class_ap_at_best_map": best_map_metrics["class_ap"]
    }

    save_json(
        summary,
        os.path.join(
            experiment_dir,
            "experiment_summary.json"
        )
    )

    print("\nExperiment completed:", loss_name)
    print("Best AUROC:", round(best_auc, 4))
    print("Best AUROC epoch:", best_auc_epoch)
    print("Best mAP:", round(best_map, 4))
    print("Best mAP epoch:", best_map_epoch)

    del model
    del optimizer
    del scheduler
    del train_loader
    del val_loader
    del criterion
    del scaler

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


# ------------------------------------------------------------
# 12. RUN ALL FOUR EXPERIMENTS
# ------------------------------------------------------------

EXPERIMENTS = [
    "bce",
    "sqrt_weighted_bce",
    "weighted_bce",
    "asymmetric_loss"
]

all_summaries = []

for experiment_name in EXPERIMENTS:

    result = run_experiment(experiment_name)

    all_summaries.append(result)

    # Save global progress after each completed experiment.
    save_json(
        all_summaries,
        os.path.join(
            OUTPUT_DIR,
            "comparison_summary.json"
        )
    )

    pd.DataFrame(all_summaries).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "comparison_summary.csv"
        ),
        index=False
    )


# ------------------------------------------------------------
# 13. FINAL COMPARISON TABLE
# ------------------------------------------------------------

comparison = pd.DataFrame(all_summaries)

comparison = comparison[
    [
        "loss_function",
        "best_auroc_epoch",
        "best_macro_auroc",
        "map_at_best_auroc",
        "micro_f1_at_best_auroc",
        "macro_f1_at_best_auroc",
        "best_map_epoch",
        "best_macro_map",
        "auroc_at_best_map"
    ]
]

comparison = comparison.sort_values(
    by="best_macro_auroc",
    ascending=False
)

print("\n" + "=" * 75)
print("FINAL LOSS FUNCTION COMPARISON")
print("=" * 75)

display(comparison)

print("\nAll experiments completed!")
print("Results saved to:", OUTPUT_DIR)


Device: cuda
Mixed precision: True


100%|██████████| 1.43G/1.43G [12:15<00:00, 1.94MB/s]



Dataset sizes:
Train: 78468
Validation: 11219

Selected classes: ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Pneumothorax']

Positive weights:
Atelectasis: 8.8134
Cardiomegaly: 39.2400
Effusion: 7.4730
Infiltration: 4.6395
Mass: 18.6760
Pneumothorax: 20.1789

Loading ImageNet pretrained ResNet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]


Common initialization prepared!

STARTING: BCE
Epoch [1/10] Train Loss: 0.2454 | Val Loss: 0.2389 | Macro AUROC: 0.8000 | Macro mAP: 0.2965 | Micro F1: 0.2330 | Macro F1: 0.1632 | LR: 0.000100
Epoch [2/10] Train Loss: 0.2274 | Val Loss: 0.2300 | Macro AUROC: 0.8118 | Macro mAP: 0.3163 | Micro F1: 0.1939 | Macro F1: 0.1627 | LR: 0.000100
Epoch [3/10] Train Loss: 0.2229 | Val Loss: 0.2273 | Macro AUROC: 0.8183 | Macro mAP: 0.3269 | Micro F1: 0.1751 | Macro F1: 0.1603 | LR: 0.000100
Epoch [4/10] Train Loss: 0.2189 | Val Loss: 0.2316 | Macro AUROC: 0.8134 | Macro mAP: 0.3275 | Micro F1: 0.2442 | Macro F1: 0.2520 | LR: 0.000100
Epoch [5/10] Train Loss: 0.2150 | Val Loss: 0.2283 | Macro AUROC: 0.8145 | Macro mAP: 0.3272 | Micro F1: 0.2057 | Macro F1: 0.1923 | LR: 0.000100
Epoch [6/10] Train Loss: 0.2039 | Val Loss: 0.2308 | Macro AUROC: 0.8205 | Macro mAP: 0.3278 | Micro F1: 0.2997 | Macro F1: 0.2643 | LR: 0.000050
Epoch [7/10] Train Loss: 0.1973 | Val Loss: 0.2320 | Macro AUROC: 0.8172 | Ma

,loss_function,best_auroc_epoch,best_macro_auroc,map_at_best_auroc,micro_f1_at_best_auroc,macro_f1_at_best_auroc,best_map_epoch,best_macro_map,auroc_at_best_map
3,asymmetric_loss,10,0.824665,0.342993,0.398288,0.386567,10,0.342993,0.824665
0,bce,6,0.820545,0.327843,0.299659,0.264334,6,0.327843,0.820545
2,weighted_bce,6,0.818185,0.322054,0.340777,0.314764,6,0.322054,0.818185
1,sqrt_weighted_bce,4,0.817851,0.319932,0.374974,0.364504,3,0.328169,0.817244



All experiments completed!
Results saved to: /content/resnet18_loss_comparison


In [ ]:
import os
import shutil

SOURCE_DIR = "/content/resnet18_loss_comparison"
DEST_DIR = "/content/drive/MyDrive/MedicalImageAI/LossComparison"

os.makedirs(os.path.dirname(DEST_DIR), exist_ok=True)

if os.path.exists(SOURCE_DIR):
    shutil.copytree(SOURCE_DIR, DEST_DIR, dirs_exist_ok=True)
    print("Results copied to Google Drive successfully!")
else:
    print("Source folder not found!")

Results copied to Google Drive successfully!


In [ ]:
import os

for root, dirs, files in os.walk(DEST_DIR):
    print(root, files)

/content/drive/MyDrive/MedicalImageAI/LossComparison ['comparison_summary.json', 'comparison_summary.csv']
/content/drive/MyDrive/MedicalImageAI/LossComparison/asymmetric_loss ['val_probabilities.npy', 'best_model_by_map.pth', 'training_history.json', 'val_probabilities_by_map.npy', 'training_history.csv', 'val_targets.npy', 'best_model.pth', 'experiment_summary.json']
/content/drive/MyDrive/MedicalImageAI/LossComparison/sqrt_weighted_bce ['val_probabilities.npy', 'best_model_by_map.pth', 'training_history.json', 'val_probabilities_by_map.npy', 'training_history.csv', 'val_targets.npy', 'best_model.pth', 'experiment_summary.json']
/content/drive/MyDrive/MedicalImageAI/LossComparison/weighted_bce ['val_probabilities.npy', 'best_model_by_map.pth', 'training_history.json', 'val_probabilities_by_map.npy', 'training_history.csv', 'val_targets.npy', 'best_model.pth', 'experiment_summary.json']
/content/drive/MyDrive/MedicalImageAI/LossComparison/bce ['val_probabilities.npy', 'best_model_by_m